#  Multi-Modal Drug Discovery — DAVIS & KIBA Datasets
### Protein Language Model + GNN-2D + GNN-3D → Affinity Prediction + Novel Drug Generation

**Dataset Support:** DAVIS (kinase inhibitors, Kd values) & KIBA (kinase inhibitor bioactivity scores)  
**Architecture:** ESM-2 PLM + GNN-2D + Pharmacophore GNN-3D → Multi-Modal Fusion → Dual Head  
**Outputs:** Binding Affinity Score + Generated Novel SMILES

###  Table of Contents
1. Install Dependencies  
2. Imports & Configuration  
3. Download DAVIS & KIBA Datasets  
4. Explore & Understand the Data  
5. Data Preprocessing  
6. Molecular Feature Engineering (2D Graph)  
7. Pharmacophore Feature Engineering (3D Graph)  
8. Protein Tokenisation (ESM-2)  
9. Dataset & DataLoader  
10. Protein Encoder (ESM-2 PLM)  
11. 2D Molecular GNN  
12. 3D Pharmacophore GNN  
13. Multi-Modal Fusion  
14. Affinity Prediction Head  
15. Novel Drug Decoder  
16. Full End-to-End Model  
17. Training Utilities  
18. Train on DAVIS  
19. Train on KIBA  
20. Evaluation & Metrics  
21. Inference: Predict Affinity + Generate Drug  
22. Visualise Results  
23. Save & Load Checkpoint  

---
##  Cell 1 — Install Dependencies
> Run this cell once. **Restart the kernel** after it completes.

In [ ]:
import subprocess, sys, torch

def pip_install(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# ── Versions for pre-built wheels ─────────────────────────────────────────────
torch_ver = torch.__version__.split('+')[0]        # e.g. 2.1.0
cuda_ver  = torch.version.cuda.replace('.', '')    # e.g. 121
pyg_url   = f'https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html'

print(f'PyTorch : {torch_ver}')
print(f'CUDA    : {cuda_ver}')
print(f'Wheel   : {pyg_url}')

# ── Core stack (already on Kaggle, fast) ──────────────────────────────────────
pip_install('scikit-learn', 'tqdm', 'lifelines', 'sentencepiece')

# ── Transformers (pinned) ─────────────────────────────────────────────────────
pip_install('transformers==4.40.0')

# ── PyG — pre-built binaries, no CUDA compilation ────────────────────────────
pip_install('torch-scatter',  '-f', pyg_url)
pip_install('torch-sparse',   '-f', pyg_url)
pip_install('torch-geometric')

# ── RDKit (pre-installed on Kaggle) ───────────────────────────────────────────
try:
    import rdkit
    print('✓ rdkit already available')
except ImportError:
    pip_install('rdkit')

print('\n Done! Restart kernel then run Cell 2.')

---
##  Cell 2 — Imports & Global Configuration

In [ ]:
# ── Suppress RDKit deprecation warnings ───────────────────────────────────────
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')          # kills ALL rdApp warnings (deprecations, errors, info)

import warnings, os
warnings.filterwarnings('ignore')       # also silences Python-level warnings
os.environ['PYTHONWARNINGS'] = 'ignore'
# ── Standard Library ──────────────────────────────────────────────────────────
import os, math, json, warnings, random, pickle
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

# ── Numerical ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# ── Progress ──────────────────────────────────────────────────────────────────
from tqdm import tqdm

# ── Cheminformatics ───────────────────────────────────────────────────────────
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, Draw
from rdkit.Chem.Draw import MolToImage

# ── Deep Learning ─────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# ── Graph Neural Networks ─────────────────────────────────────────────────────
from torch_geometric.data import Data, Batch
from torch_geometric.nn import (
    GATConv, GCNConv, GINConv,
    global_mean_pool, global_add_pool, global_max_pool
)

# ── Protein Language Model ────────────────────────────────────────────────────
from transformers import AutoTokenizer, EsmModel

# ── Metrics ───────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
from lifelines.utils import concordance_index

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'  Device  : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU     : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'  PyTorch : {torch.__version__}')
print(f'  Seed    : {SEED}')

---
##  Cell  — Hyperparameters & Config
> All tunable parameters live here — change once, applies everywhere.

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET          = 'davis'        # 'davis' or 'kiba'
DATA_DIR         = Path('./data')
DATA_DIR.mkdir(exist_ok=True)

# ── Protein Encoder ───────────────────────────────────────────────────────────
# FIX: upgraded from 8M → 35M model; unfreezing PLM so it adapts to affinity task
PLM_NAME         = 'facebook/esm2_t12_35M_UR50D'  # was esm2_t6_8M_UR50D
MAX_PROT_LEN     = 512
FREEZE_PLM       = False           # FIX: was True — frozen PLM learned nothing

# ── GNN ───────────────────────────────────────────────────────────────────────
ATOM_FEAT_DIM    = 55
BOND_FEAT_DIM    = 6
PHARM_FEAT_DIM   = 8
GNN_HIDDEN       = 128
GNN_OUT          = 256
GNN_LAYERS       = 3
GAT_HEADS        = 4

# ── Fusion & Heads ────────────────────────────────────────────────────────────
MODAL_DIM        = 256
FUSION_DIM       = 512
FUSION_HEADS     = 8
FUSION_LAYERS    = 2

# ── Decoder ───────────────────────────────────────────────────────────────────
MAX_SMILES_LEN   = 120
DECODER_LAYERS   = 3
DECODER_HEADS    = 8

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS           = 50
BATCH_SIZE       = 16
LR               = 3e-4
WEIGHT_DECAY     = 1e-5
GRAD_CLIP        = 1.0
# FIX: ALPHA raised to 0.9 so affinity loss dominates in the combined objective.
# Generation loss is ~20x larger in raw magnitude, so even with alpha=0.9 the
# gradient balance is healthier. Raise further (→ 1.0) to train affinity-only.
ALPHA            = 0.9             # was 0.7
VAL_SPLIT        = 0.1
TEST_SPLIT       = 0.1

# ── Output ────────────────────────────────────────────────────────────────────
CKPT_DIR         = Path('./checkpoints')
CKPT_DIR.mkdir(exist_ok=True)

print('  Configuration loaded:')
print(f'   Dataset     : {DATASET.upper()}')
print(f'   PLM         : {PLM_NAME}')
print(f'   Freeze PLM  : {FREEZE_PLM}')
print(f'   Batch size  : {BATCH_SIZE}')
print(f'   Epochs      : {EPOCHS}')
print(f'   Alpha       : {ALPHA}')
print(f'   Fusion dim  : {FUSION_DIM}')
print(f'   Device      : {DEVICE}')


---
##  Cell 4 — Download DAVIS & KIBA Datasets
> We use the widely-cited versions from the DeepDTA / GraphDTA paper repositories.

In [ ]:
import urllib.request, zipfile, gzip, shutil

# ── DAVIS ─────────────────────────────────────────────────────────────────────
DAVIS_URL = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data/davis/'
KIBA_URL  = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data/kiba/'

FILES = {
    'davis': ['ligands_can.txt', 'proteins.txt', 'Y', 'folds/train_fold_setting1.txt', 'folds/test_fold_setting1.txt'],
    'kiba':  ['ligands_can.txt', 'proteins.txt', 'Y', 'folds/train_fold_setting1.txt', 'folds/test_fold_setting1.txt'],
}

def download_file(url, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f'  ✓ {dest.name} already exists')
        return
    urllib.request.urlretrieve(url, dest)
    print(f'  ↓ Downloaded: {dest.name}')

for dataset, files in FILES.items():
    base_url = DAVIS_URL if dataset == 'davis' else KIBA_URL
    print(f'\n📂 {dataset.upper()}:')
    for f in files:
        download_file(base_url + f, DATA_DIR / dataset / f)

print('\n All dataset files downloaded!')

---
## 🔍 Cell 5 — Load & Parse the Dataset

In [ ]:
import json

def load_dataset(name: str):
    """
    Returns a DataFrame with columns:
      smiles, protein_seq, affinity, split ('train'/'test')
    """
    base = DATA_DIR / name

    # --- Load drug SMILES ---
    with open(base / 'ligands_can.txt') as f:
        drugs_raw = json.load(f)   # dict: name -> canonical SMILES
    drugs = list(drugs_raw.values())   # list of SMILES strings

    # --- Load protein sequences ---
    with open(base / 'proteins.txt') as f:
        prots_raw = json.load(f)   # dict: name -> AA sequence
    proteins = list(prots_raw.values())

    # --- Load affinity matrix ---
# Y file is pickle binary in DeepDTA repo, not JSON
    import pickle
    with open(base / 'Y', 'rb') as f:       # 'rb' = read binary
     Y_raw = pickle.load(f, encoding='latin1')
    Y = np.array(Y_raw)

    # --- Load folds ---
    with open(base / 'folds/train_fold_setting1.txt') as f:
        train_folds = json.load(f)   # list of lists
    with open(base / 'folds/test_fold_setting1.txt') as f:
        test_fold = json.load(f)     # list of ints

    train_idx = [idx for fold in train_folds for idx in fold]
    test_idx  = list(test_fold)

    # Build flat (drug_idx, protein_idx) pair list
    rows = []
    n_drugs, n_prots = Y.shape
    for d in range(n_drugs):
        for p in range(n_prots):
            val = Y[d, p]
            if not np.isnan(val):
                rows.append({'drug_idx': d, 'protein_idx': p,
                             'smiles': drugs[d], 'protein_seq': proteins[p],
                             'affinity': float(val)})

    df = pd.DataFrame(rows)

    # Map split: use drug index for train/test assignment
    train_set = set(train_idx)
    test_set  = set(test_idx)
    df['split'] = df['drug_idx'].apply(
        lambda x: 'train' if x in train_set else ('test' if x in test_set else 'val')
    )

    return df


df = load_dataset(DATASET)

print(f' {DATASET.upper()} Dataset Loaded')
print(f'   Total pairs  : {len(df):,}')
print(f'   Train pairs  : {(df.split=="train").sum():,}')
print(f'   Test pairs   : {(df.split=="test").sum():,}')
print(f'   Val pairs    : {(df.split=="val").sum():,}')
print(f'   Unique drugs : {df.smiles.nunique():,}')
print(f'   Unique prots : {df.protein_seq.nunique():,}')
print(f'   Affinity range: [{df.affinity.min():.3f}, {df.affinity.max():.3f}]')
df.head()

---
##  Cell 6 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'{DATASET.upper()} — Dataset Overview', fontsize=14, fontweight='bold')

# Affinity distribution
axes[0].hist(df['affinity'], bins=60, color='#3b82f6', edgecolor='white', alpha=0.85)
axes[0].set_title('Affinity Distribution')
axes[0].set_xlabel('Affinity (pKd)' if DATASET == 'davis' else 'KIBA Score')
axes[0].set_ylabel('Count')
axes[0].axvline(df['affinity'].mean(), color='#ef4444', lw=2, linestyle='--', label=f'Mean={df["affinity"].mean():.2f}')
axes[0].legend()

# SMILES length distribution
df['smiles_len'] = df['smiles'].apply(len)
axes[1].hist(df['smiles_len'], bins=40, color='#10b981', edgecolor='white', alpha=0.85)
axes[1].set_title('SMILES Length Distribution')
axes[1].set_xlabel('SMILES Length')
axes[1].set_ylabel('Count')

# Protein sequence length
df['prot_len'] = df['protein_seq'].apply(len)
axes[2].hist(df['prot_len'], bins=40, color='#f59e0b', edgecolor='white', alpha=0.85)
axes[2].set_title('Protein Sequence Length')
axes[2].set_xlabel('Sequence Length')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSMILES length  — mean: {df.smiles_len.mean():.0f}, max: {df.smiles_len.max()}')
print(f'Protein length — mean: {df.prot_len.mean():.0f}, max: {df.prot_len.max()}')

---
##  Cell 7 — Affinity Normalisation
> DAVIS uses Kd values (log-transform); KIBA uses composite bioactivity scores.

In [ ]:
def normalise_affinity(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    df = df.copy()
    if dataset == 'davis':
        # Convert Kd → pKd = -log10(Kd/1e9); DAVIS stores Kd in nM already
        # Some implementations use: pKd = -log10(Kd * 1e-9)
        # The stored values are already -log10 transformed in DeepDTA's version.
        # We normalise to [0, 1] for stable training.
        aff = df['affinity'].values
        df['affinity_norm'] = (aff - aff.min()) / (aff.max() - aff.min())
        df['affinity_raw']  = aff
    elif dataset == 'kiba':
        # KIBA scores: normalise to [0, 1]
        aff = df['affinity'].values
        df['affinity_norm'] = (aff - aff.min()) / (aff.max() - aff.min())
        df['affinity_raw']  = aff
    return df

df = normalise_affinity(df, DATASET)

# ── Validate SMILES ────────────────────────────────────────────────────────────
print('Validating SMILES...')
def is_valid_smiles(smi):
    return Chem.MolFromSmiles(smi) is not None

df['valid_smiles'] = df['smiles'].apply(is_valid_smiles)
invalid = (~df['valid_smiles']).sum()
print(f'  Invalid SMILES dropped: {invalid}')
df = df[df['valid_smiles']].copy()

# ── Validate proteins ──────────────────────────────────────────────────────────
VALID_AAS = set('ACDEFGHIKLMNPQRSTVWY')
df['valid_prot'] = df['protein_seq'].apply(lambda s: all(c in VALID_AAS for c in s))
invalid_p = (~df['valid_prot']).sum()
print(f'  Invalid proteins dropped: {invalid_p}')
df = df[df['valid_prot']].copy()

print(f'\n Clean pairs remaining: {len(df):,}')
print(f'   Normalised affinity range: [{df.affinity_norm.min():.3f}, {df.affinity_norm.max():.3f}]')

---
## ⚗️ Cell 8 — Atom & Bond Feature Functions (2D Graph)

In [ ]:
from rdkit.Chem import rdchem

ALLOWABLE_ATOMS = [
    'C','N','O','S','F','P','Cl','Br','I','B','Si','Se',
    'Te','Sn','Pb','As','Sb','Bi','unknown'
]

def one_hot(value, choices):
    """One-hot encode value against choices list; last slot = 'other'."""
    enc = [0] * len(choices)
    if value in choices:
        enc[choices.index(value)] = 1
    else:
        enc[-1] = 1
    return enc

def atom_features(atom) -> list:
    """
    78-dimensional atom feature vector:
     - atom type one-hot (19)
     - degree (11)
     - implicit valence (11)
     - formal charge (1)
     - num radical electrons (1)
     - hybridisation (6)
     - is aromatic (1)
     - total num Hs (5)
     - chirality (4)
    """
    return (
        one_hot(atom.GetSymbol(), ALLOWABLE_ATOMS) +
        one_hot(atom.GetDegree(), [0,1,2,3,4,5,6,7,8,9,10]) +
        one_hot(atom.GetImplicitValence(), [0,1,2,3,4,5,10]) +
        [atom.GetFormalCharge()] +
        [atom.GetNumRadicalElectrons()] +
        one_hot(str(atom.GetHybridization()), [
            'SP','SP2','SP3','SP3D','SP3D2','other'
        ]) +
        [int(atom.GetIsAromatic())] +
        one_hot(atom.GetTotalNumHs(), [0,1,2,3,4]) +
        one_hot(str(atom.GetChiralTag()), [
            'CHI_UNSPECIFIED','CHI_TETRAHEDRAL_CW',
            'CHI_TETRAHEDRAL_CCW','CHI_OTHER'
        ])
    )

def bond_features(bond) -> list:
    """
    6-dimensional bond feature vector.
    """
    bt = bond.GetBondTypeAsDouble()
    return [
        int(bt == 1.0),   # single
        int(bt == 2.0),   # double
        int(bt == 3.0),   # triple
        int(bt == 1.5),   # aromatic
        int(bond.GetIsConjugated()),
        int(bond.IsInRing())
    ]

# Verify dimension
from rdkit import Chem
_mol = Chem.MolFromSmiles('CC(=O)O')
_atom_dim = len(atom_features(_mol.GetAtomWithIdx(0)))
_bond_dim = len(bond_features(_mol.GetBondWithIdx(0)))
ATOM_FEAT_DIM = _atom_dim   # auto-set to whatever RDKit returns
BOND_FEAT_DIM = _bond_dim
print(f' Atom feature dim : {_atom_dim}')
print(f' Bond feature dim : {_bond_dim}')

---
##  Cell 9 — SMILES → 2D Molecular Graph

In [ ]:
def smiles_to_graph_2d(smiles: str) -> Data | None:
    """
    Converts a SMILES string into a PyG Data object with:
      x          : (N, 78)  atom feature matrix
      edge_index : (2, 2*E) bidirectional edge list
      edge_attr  : (2*E, 6) bond feature matrix
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Atom features
    x = torch.tensor(
        [atom_features(a) for a in mol.GetAtoms()],
        dtype=torch.float
    )  # (N, 78)

    # Edge features (bidirectional)
    src, dst, ea = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        src += [i, j]; dst += [j, i]
        ea  += [bf, bf]

    if len(src) == 0:  # single atom molecule
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_FEAT_DIM))
    else:
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        edge_attr  = torch.tensor(ea, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


# ── Test ──────────────────────────────────────────────────────────────────────
test_smiles = df['smiles'].iloc[0]
g = smiles_to_graph_2d(test_smiles)
print(f'Test SMILES : {test_smiles}')
print(f'  Nodes     : {g.num_nodes}')
print(f'  Edges     : {g.num_edges}')
print(f'  x shape   : {g.x.shape}')
print(f'  edge_attr : {g.edge_attr.shape}')

---
##  Cell 10 — SMILES → 3D Pharmacophore Graph

In [ ]:
DISTANCE_THRESHOLD = 4.5   # Angstroms — spatial edge cutoff

def smiles_to_pharmacophore_graph(smiles: str) -> Data | None:
    """
    Generates a 3D pharmacophore graph from a SMILES string.
    Node features (8-dim):
      atomic_num, is_aromatic, is_hbond_acceptor, is_hbond_donor,
      is_hydrophobic, pos_charge_proxy, neg_charge_proxy, ring_membership
    Edges: spatial distance < DISTANCE_THRESHOLD
    Edge attr: distance (1-dim)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    mol = Chem.AddHs(mol)
    try:
        res = AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
        if res == -1:
            AllChem.EmbedMolecule(mol, randomSeed=SEED)
        AllChem.MMFFOptimizeMolecule(mol, maxIters=200)
    except Exception:
        try:
            AllChem.EmbedMolecule(mol, randomSeed=SEED)
        except Exception:
            return None

    try:
        conf = mol.GetConformer()
    except ValueError:
        return None

    n = mol.GetNumAtoms()
    positions = np.array([list(conf.GetAtomPosition(i)) for i in range(n)])

    # Pharmacophore node features
    ring_info = mol.GetRingInfo()
    feats = []
    for atom in mol.GetAtoms():
        sym = atom.GetSymbol()
        feats.append([
            atom.GetAtomicNum() / 100.0,
            int(atom.GetIsAromatic()),
            int(sym in ('O', 'N', 'F', 'S')),       # H-acceptor proxy
            int(sym in ('N', 'O') and atom.GetTotalNumHs() > 0),  # H-donor
            int(sym in ('C',) and atom.GetIsAromatic()),           # hydrophobic
            int(atom.GetFormalCharge() > 0),
            int(atom.GetFormalCharge() < 0),
            int(ring_info.NumAtomRings(atom.GetIdx()) > 0)
        ])
    x = torch.tensor(feats, dtype=torch.float)  # (N, 8)

    # Spatial edges
    src, dst, dists = [], [], []
    for i in range(n):
        for j in range(i + 1, n):
            dist = np.linalg.norm(positions[i] - positions[j])
            if dist < DISTANCE_THRESHOLD:
                src += [i, j]; dst += [j, i]
                dists += [dist, dist]

    if len(src) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, 1))
    else:
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        edge_attr  = torch.tensor(dists, dtype=torch.float).unsqueeze(1)

    pos = torch.tensor(positions, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, pos=pos)


# ── Test ──────────────────────────────────────────────────────────────────────
pg = smiles_to_pharmacophore_graph(test_smiles)
if pg:
    print(f'Pharmacophore graph — nodes: {pg.num_nodes}, edges: {pg.num_edges}')
    print(f'  x shape  : {pg.x.shape}')
    print(f'  pos shape: {pg.pos.shape}')
else:
    print('  3D conformer failed — this molecule will be skipped')

---
##  Cell 11 — SMILES Vocabulary (for Decoder)

In [ ]:
# Build vocabulary from ALL unique SMILES in the dataset
_all_chars = set()
for smi in df['smiles'].unique():
    _all_chars.update(smi)

SPECIAL_TOKENS = ['<PAD>', '<SOS>', '<EOS>']
SMILES_CHARS   = SPECIAL_TOKENS + sorted(_all_chars)

CHAR2IDX = {c: i for i, c in enumerate(SMILES_CHARS)}
IDX2CHAR = {i: c for c, i in CHAR2IDX.items()}
VOCAB_SIZE = len(CHAR2IDX)

PAD_IDX = CHAR2IDX['<PAD>']
SOS_IDX = CHAR2IDX['<SOS>']
EOS_IDX = CHAR2IDX['<EOS>']

def encode_smiles(smiles: str, max_len: int = MAX_SMILES_LEN) -> torch.Tensor:
    tokens = [SOS_IDX]
    for c in smiles[:max_len - 2]:
        tokens.append(CHAR2IDX.get(c, PAD_IDX))
    tokens.append(EOS_IDX)
    # Pad
    tokens += [PAD_IDX] * (max_len - len(tokens))
    return torch.tensor(tokens[:max_len], dtype=torch.long)

def decode_smiles(token_ids) -> str:
    result = []
    for idx in token_ids:
        idx = idx.item() if hasattr(idx, 'item') else int(idx)
        if idx == EOS_IDX: break
        if idx in (SOS_IDX, PAD_IDX): continue
        result.append(IDX2CHAR.get(idx, ''))
    return ''.join(result)

print(f' Vocabulary size : {VOCAB_SIZE}')
print(f'   PAD={PAD_IDX}, SOS={SOS_IDX}, EOS={EOS_IDX}')

# Sanity check
_enc = encode_smiles(test_smiles)
_dec = decode_smiles(_enc)
print(f'   Original : {test_smiles[:50]}')
print(f'   Decoded  : {_dec[:50]}')
assert test_smiles[:MAX_SMILES_LEN - 2] == _dec[:MAX_SMILES_LEN - 2], 'Encode/decode mismatch!'
print('   Round-trip ')

---
##  Cell 12 — Dataset Class

In [ ]:
class DrugTargetDataset(Dataset):
    """
    Returns one sample per drug-protein pair.
    Each sample:
      graph_2d  : PyG Data (2D molecular graph)
      graph_3d  : PyG Data (3D pharmacophore graph)
      prot_ids  : LongTensor (MAX_PROT_LEN,)
      prot_mask : LongTensor (MAX_PROT_LEN,)
      affinity  : FloatTensor scalar (normalised)
      smiles_enc: LongTensor (MAX_SMILES_LEN,)  — for decoder teacher forcing
    """

    def __init__(self, dataframe: pd.DataFrame, tokeniser, max_prot_len: int = MAX_PROT_LEN):
        self.df           = dataframe.reset_index(drop=True)
        self.tokeniser    = tokeniser
        self.max_prot_len = max_prot_len
        self._graph_cache = {}   # cache graphs to avoid recomputing

    def __len__(self):
        return len(self.df)

    def _get_graphs(self, smiles):
        if smiles not in self._graph_cache:
            g2d = smiles_to_graph_2d(smiles)
            g3d = smiles_to_pharmacophore_graph(smiles)
            self._graph_cache[smiles] = (g2d, g3d)
        return self._graph_cache[smiles]

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        smiles  = row['smiles']
        protein = row['protein_seq']

        # ── Graphs ──────────────────────────────────────────────────────────
        graph_2d, graph_3d = self._get_graphs(smiles)

        # ── Protein tokens ──────────────────────────────────────────────────
        enc = self.tokeniser(
            protein[:self.max_prot_len],
            max_length=self.max_prot_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        prot_ids  = enc['input_ids'].squeeze(0)
        prot_mask = enc['attention_mask'].squeeze(0)

        # ── SMILES encoding (decoder target) ────────────────────────────────
        smiles_enc = encode_smiles(smiles)

        # ── Affinity ────────────────────────────────────────────────────────
        affinity = torch.tensor(row['affinity_norm'], dtype=torch.float)

        return {
            'graph_2d':   graph_2d,
            'graph_3d':   graph_3d,
            'prot_ids':   prot_ids,
            'prot_mask':  prot_mask,
            'smiles_enc': smiles_enc,
            'affinity':   affinity,
            'smiles':     smiles,
        }


def collate_fn(batch):
    # Filter out None graphs (failed 3D conformer)
    batch = [b for b in batch if b['graph_2d'] is not None and b['graph_3d'] is not None]
    if not batch:
        return None

    return {
        'graph_2d':   Batch.from_data_list([b['graph_2d']   for b in batch]),
        'graph_3d':   Batch.from_data_list([b['graph_3d']   for b in batch]),
        'prot_ids':   torch.stack([b['prot_ids']   for b in batch]),
        'prot_mask':  torch.stack([b['prot_mask']  for b in batch]),
        'smiles_enc': torch.stack([b['smiles_enc'] for b in batch]),
        'affinity':   torch.stack([b['affinity']   for b in batch]),
        'smiles':     [b['smiles'] for b in batch],
    }

print('✅ DrugTargetDataset and collate_fn defined.')

---
##  Cell 13 — Build DataLoaders

In [ ]:
print(f'Loading ESM-2 tokeniser: {PLM_NAME}')
tokeniser = AutoTokenizer.from_pretrained(PLM_NAME)
print(' Tokeniser ready.')

train_df = df[df['split'] == 'train'].reset_index(drop=True)
test_df  = df[df['split'] == 'test' ].reset_index(drop=True)
val_df   = df[df['split'] == 'val'  ].reset_index(drop=True)

# FIX: random val split instead of sequential slice (which caused distribution mismatch)
if len(val_df) == 0:
    val_size  = int(0.1 * len(train_df))
    val_idx   = train_df.sample(val_size, random_state=SEED).index
    val_df    = train_df.loc[val_idx].copy()
    train_df  = train_df.drop(val_idx).copy()

print(f'\nSplits — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

train_dataset = DrugTargetDataset(train_df, tokeniser)
val_dataset   = DrugTargetDataset(val_df,   tokeniser)
test_dataset  = DrugTargetDataset(test_df,  tokeniser)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True,  collate_fn=collate_fn,
    num_workers=2, pin_memory=(DEVICE.type == 'cuda')
)
val_loader = DataLoader(
    val_dataset,   batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn,
    num_workers=2, pin_memory=(DEVICE.type == 'cuda')
)
test_loader = DataLoader(
    test_dataset,  batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn,
    num_workers=2, pin_memory=(DEVICE.type == 'cuda')
)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

# Verify one batch
sample_batch = next(iter(train_loader))
print(f'\nSample batch keys : {list(sample_batch.keys())}')
print(f'  prot_ids shape  : {sample_batch["prot_ids"].shape}')
print(f'  affinity shape  : {sample_batch["affinity"].shape}')
print(f'  graph_2d nodes  : {sample_batch["graph_2d"].num_nodes}')


---
##  Cell 14 — Protein Encoder (ESM-2 PLM)

In [ ]:
class ProteinEncoder(nn.Module):
    """
    ESM-2 Protein Language Model encoder.
    - Optionally frozen (FREEZE_PLM=True) or fine-tuned
    - CLS token → Linear projection → modal_dim
    """

    def __init__(
        self,
        model_name: str  = PLM_NAME,
        modal_dim:  int  = MODAL_DIM,
        freeze:     bool = FREEZE_PLM,
        dropout:    float = 0.1
    ):
        super().__init__()
        print(f'  Loading ESM-2 weights from {model_name}...')
        self.esm     = EsmModel.from_pretrained(model_name)
        self.esm_dim = self.esm.config.hidden_size  # 320 for 8M

        if freeze:
            for p in self.esm.parameters():
                p.requires_grad = False
            print(f'  ESM-2 frozen ({sum(p.numel() for p in self.esm.parameters()):,} params frozen)')

        self.proj = nn.Sequential(
            nn.LayerNorm(self.esm_dim),
            nn.Linear(self.esm_dim, modal_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(modal_dim, modal_dim)
        )

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        input_ids, attention_mask: (B, L)
        Returns: (B, modal_dim)
        """
        out = self.esm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        # Mean pool over non-padding positions
        token_emb  = out.last_hidden_state       # (B, L, esm_dim)
        mask_exp   = attention_mask.unsqueeze(-1).float()
        mean_emb   = (token_emb * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1)
        return self.proj(mean_emb)               # (B, modal_dim)


# Quick test
_pe = ProteinEncoder()
_ids  = sample_batch['prot_ids'][:2]
_mask = sample_batch['prot_mask'][:2]
_out  = _pe(_ids, _mask)
print(f'\n ProteinEncoder output: {_out.shape}')  # should be (2, 256)
del _pe, _out

---
##  Cell 15 — 2D Molecular GNN (GAT)

In [ ]:
class MolGNN2D(nn.Module):
    """
    Multi-layer Graph Attention Network for 2D molecular graphs.
    Input node features: (N, ATOM_FEAT_DIM=78)
    Output: (B, modal_dim) via global mean pooling
    """

    def __init__(
        self,
        in_dim:     int  = ATOM_FEAT_DIM,
        hidden_dim: int  = GNN_HIDDEN,
        out_dim:    int  = GNN_OUT,
        n_layers:   int  = GNN_LAYERS,
        heads:      int  = GAT_HEADS,
        dropout:    float = 0.1
    ):
        super().__init__()
        self.dropout = dropout

        # Input projection
        self.input_proj = nn.Linear(in_dim, hidden_dim * heads)

        # GAT layers
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for i in range(n_layers - 1):
            self.convs.append(
                GATConv(hidden_dim * heads, hidden_dim, heads=heads, concat=True, dropout=dropout)
            )
            self.norms.append(nn.LayerNorm(hidden_dim * heads))

        # Final layer — collapse multi-head
        self.convs.append(
            GATConv(hidden_dim * heads, out_dim, heads=1, concat=False, dropout=dropout)
        )
        self.norms.append(nn.LayerNorm(out_dim))

        # Output projection
        self.out_proj = nn.Sequential(
            nn.Linear(out_dim, out_dim),
            nn.GELU()
        )

    def forward(self, data: Data) -> torch.Tensor:
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Project input
        x = F.gelu(self.input_proj(x))

        # GAT message passing
        for conv, norm in zip(self.convs, self.norms):
            residual = x if x.shape == conv(x, edge_index).shape else None
            x_new = F.gelu(norm(conv(x, edge_index)))
            x_new = F.dropout(x_new, p=self.dropout, training=self.training)
            # Residual when dimensions match
            if residual is not None and x_new.shape == residual.shape:
                x_new = x_new + residual
            x = x_new

        # Global pooling
        x = global_mean_pool(x, batch)   # (B, out_dim)
        return self.out_proj(x)


# Test
_gnn2d = MolGNN2D()
_g2d   = sample_batch['graph_2d']
_out2d = _gnn2d(_g2d)
print(f' MolGNN2D output: {_out2d.shape}')  # (B, 256)
del _gnn2d, _out2d

---
##  Cell 16 — 3D Pharmacophore GNN (GIN)

In [ ]:
class PharmacoGNN3D(nn.Module):
    """
    Graph Isomorphism Network on the 3D pharmacophore spatial graph.
    GIN is more expressive than GCN for structural features.
    Input node features: (N, PHARM_FEAT_DIM=8)
    Output: (B, modal_dim)
    """

    def __init__(
        self,
        in_dim:     int  = PHARM_FEAT_DIM,
        hidden_dim: int  = GNN_HIDDEN,
        out_dim:    int  = GNN_OUT,
        n_layers:   int  = GNN_LAYERS,
        dropout:    float = 0.1
    ):
        super().__init__()
        self.dropout = dropout

        self.input_proj = nn.Linear(in_dim, hidden_dim)

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        for _ in range(n_layers - 1):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.BatchNorm1d(hidden_dim * 2),
                nn.ReLU(),
                nn.Linear(hidden_dim * 2, hidden_dim)
            )
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        # Final GIN layer → out_dim
        mlp_last = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, out_dim)
        )
        self.convs.append(GINConv(mlp_last))
        self.bns.append(nn.BatchNorm1d(out_dim))

        self.out_proj = nn.Sequential(
            nn.Linear(out_dim, out_dim),
            nn.GELU()
        )

    def forward(self, data: Data) -> torch.Tensor:
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.relu(self.input_proj(x))

        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Concatenate mean + max pooling for richer representation
        x_mean = global_mean_pool(x, batch)  # (B, out_dim)
        x_max  = global_max_pool(x,  batch)  # (B, out_dim)
        x = (x_mean + x_max) / 2.0
        return self.out_proj(x)


# Test
_gnn3d = PharmacoGNN3D()
_g3d   = sample_batch['graph_3d']
_out3d = _gnn3d(_g3d)
print(f' PharmacoGNN3D output: {_out3d.shape}')  # (B, 256)
del _gnn3d, _out3d

---
##  Cell 17 — Multi-Modal Fusion (Cross-Attention Transformer)

In [ ]:
class MultiModalFusion(nn.Module):
    """
    Fuses three modality embeddings using a Transformer encoder.

    Strategy:
      1. Project each modality to fusion_dim/3 then concat → fusion_dim
      2. Treat each modality as a 'token' → shape (B, 3, fusion_dim)
      3. Transformer encoder learns cross-modal interactions
      4. Mean-pool the 3 token outputs → (B, fusion_dim)
    """

    def __init__(
        self,
        modal_dim:  int = MODAL_DIM,
        fusion_dim: int = FUSION_DIM,
        n_heads:    int = FUSION_HEADS,
        n_layers:   int = FUSION_LAYERS,
        dropout:    float = 0.1
    ):
        super().__init__()
        self.fusion_dim = fusion_dim

        # Per-modality projections
        self.proj_prot = nn.Linear(modal_dim, fusion_dim)
        self.proj_2d   = nn.Linear(modal_dim, fusion_dim)
        self.proj_3d   = nn.Linear(modal_dim, fusion_dim)

        # Learnable modality-type embeddings
        self.modality_emb = nn.Embedding(3, fusion_dim)

        # Transformer encoder (3 tokens × fusion_dim)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=fusion_dim,
            nhead=n_heads,
            dim_feedforward=fusion_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True          # Pre-LN for stability
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.out_norm = nn.LayerNorm(fusion_dim)

    def forward(
        self,
        prot_emb:  torch.Tensor,    # (B, modal_dim)
        mol2d_emb: torch.Tensor,    # (B, modal_dim)
        mol3d_emb: torch.Tensor,    # (B, modal_dim)
    ) -> torch.Tensor:

        p = self.proj_prot(prot_emb)   # (B, fusion_dim)
        m = self.proj_2d(mol2d_emb)
        q = self.proj_3d(mol3d_emb)

        # Add modality embeddings
        mod_idx = torch.arange(3, device=p.device)
        mod_emb = self.modality_emb(mod_idx)   # (3, fusion_dim)
        p = p + mod_emb[0]
        m = m + mod_emb[1]
        q = q + mod_emb[2]

        # Stack → (B, 3, fusion_dim)
        tokens = torch.stack([p, m, q], dim=1)

        # Cross-modal attention
        fused = self.transformer(tokens)    # (B, 3, fusion_dim)

        # Aggregate
        fused = fused.mean(dim=1)           # (B, fusion_dim)
        return self.out_norm(fused)


# Test
_fusion  = MultiModalFusion()
_p = torch.randn(4, MODAL_DIM)
_m = torch.randn(4, MODAL_DIM)
_q = torch.randn(4, MODAL_DIM)
_fused = _fusion(_p, _m, _q)
print(f' MultiModalFusion output: {_fused.shape}')  # (4, 512)
del _fusion, _fused

---
##  Cell 18 — Affinity Prediction Head

In [ ]:
class AffinityHead(nn.Module):
    """
    Regression MLP: fused embedding → scalar affinity score.
    Outputs are in [0, 1] (matching normalised labels).
    """

    def __init__(self, fusion_dim: int = FUSION_DIM, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, 1),
            nn.Sigmoid()          # output in [0, 1]
        )

    def forward(self, fused: torch.Tensor) -> torch.Tensor:
        return self.net(fused).squeeze(-1)   # (B,)


print(' AffinityHead defined.')

---
##  Cell 19 — Novel Drug Decoder (Transformer)

In [ ]:
class DrugDecoder(nn.Module):
    """
    Autoregressive Transformer decoder that generates novel SMILES
    conditioned on the fused protein+molecule embedding.

    Training : teacher forcing (full target sequence passed as input)
    Inference: greedy / temperature sampling
    """

    def __init__(
        self,
        vocab_size:  int = VOCAB_SIZE,
        fusion_dim:  int = FUSION_DIM,
        max_len:     int = MAX_SMILES_LEN,
        n_layers:    int = DECODER_LAYERS,
        n_heads:     int = DECODER_HEADS,
        dropout:     float = 0.1
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.fusion_dim = fusion_dim
        self.max_len    = max_len

        self.tok_emb  = nn.Embedding(vocab_size, fusion_dim, padding_idx=PAD_IDX)
        self.pos_emb  = nn.Embedding(max_len + 4, fusion_dim)
        self.drop     = nn.Dropout(dropout)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=fusion_dim,
            nhead=n_heads,
            dim_feedforward=fusion_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.decoder  = nn.TransformerDecoder(dec_layer, num_layers=n_layers)
        self.out_proj = nn.Linear(fusion_dim, vocab_size)

    def forward(
        self,
        fused:      torch.Tensor,   # (B, fusion_dim) — encoder memory
        tgt_tokens: torch.Tensor    # (B, T)          — teacher-forced targets
    ) -> torch.Tensor:              # (B, T, vocab_size)

        B, T = tgt_tokens.shape
        pos  = torch.arange(T, device=tgt_tokens.device).unsqueeze(0).expand(B, -1)

        tgt_emb = self.drop(self.tok_emb(tgt_tokens) + self.pos_emb(pos))

        # Causal mask
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            T, device=tgt_tokens.device
        ).bool()

        # Padding mask
        tgt_key_pad = (tgt_tokens == PAD_IDX)

        # Memory: (B, 1, fusion_dim)
        memory = fused.unsqueeze(1)

        out = self.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_pad
        )  # (B, T, fusion_dim)

        return self.out_proj(out)  # (B, T, vocab_size)

    @torch.no_grad()
    def generate(
        self,
        fused:       torch.Tensor,
        temperature: float = 1.0,
        max_len:     int   = None
    ) -> torch.Tensor:
        """Autoregressive token-by-token generation."""
        max_len = max_len or self.max_len
        B = fused.size(0)
        generated = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=fused.device)
        done = torch.zeros(B, dtype=torch.bool, device=fused.device)

        for _ in range(max_len):
            logits  = self.forward(fused, generated)[:, -1, :]  # (B, V)
            probs   = F.softmax(logits / max(temperature, 1e-5), dim=-1)
            next_tok = torch.multinomial(probs, 1)               # (B, 1)
            generated = torch.cat([generated, next_tok], dim=1)
            done |= (next_tok.squeeze(1) == EOS_IDX)
            if done.all(): break

        return generated[:, 1:]  # strip SOS


print(' DrugDecoder defined.')

---
## Cell 20 — Full End-to-End Model

In [ ]:
class MultiModalDrugDiscovery(nn.Module):
    """
    Complete pipeline:
      [ProteinEncoder | MolGNN2D | PharmacoGNN3D]
              ↓
        MultiModalFusion
              ↓
      [AffinityHead | DrugDecoder]
    """

    def __init__(self):
        super().__init__()
        self.prot_encoder  = ProteinEncoder()
        self.gnn_2d        = MolGNN2D()
        self.gnn_3d        = PharmacoGNN3D()
        self.fusion        = MultiModalFusion()
        self.affinity_head = AffinityHead()
        self.drug_decoder  = DrugDecoder()

    def forward(
        self,
        graph_2d:    Data,
        graph_3d:    Data,
        prot_ids:    torch.Tensor,
        prot_mask:   torch.Tensor,
        smiles_tgt:  torch.Tensor = None,   # (B, T) for decoder teacher forcing
    ):
        # ── Encode ──────────────────────────────────────────────────────────
        prot_emb  = self.prot_encoder(prot_ids, prot_mask)     # (B, MODAL_DIM)
        mol2d_emb = self.gnn_2d(graph_2d)                      # (B, MODAL_DIM)
        mol3d_emb = self.gnn_3d(graph_3d)                      # (B, MODAL_DIM)

        # ── Fuse ────────────────────────────────────────────────────────────
        fused = self.fusion(prot_emb, mol2d_emb, mol3d_emb)    # (B, FUSION_DIM)

        # ── Head 1: Affinity ────────────────────────────────────────────────
        aff_pred = self.affinity_head(fused)                    # (B,)

        # ── Head 2: Drug Generation ─────────────────────────────────────────
        smi_logits = None
        if smiles_tgt is not None:
            smi_logits = self.drug_decoder(fused, smiles_tgt)   # (B, T, V)

        return aff_pred, smi_logits, fused

    @torch.no_grad()
    def predict_and_generate(
        self,
        graph_2d, graph_3d,
        prot_ids, prot_mask,
        temperature: float = 0.8
    ):
        prot_emb  = self.prot_encoder(prot_ids, prot_mask)
        mol2d_emb = self.gnn_2d(graph_2d)
        mol3d_emb = self.gnn_3d(graph_3d)
        fused     = self.fusion(prot_emb, mol2d_emb, mol3d_emb)
        aff       = self.affinity_head(fused)
        gen_toks  = self.drug_decoder.generate(fused, temperature=temperature)
        return aff, gen_toks


# ── Instantiate & summarise ──────────────────────────────────────────────────
model = MultiModalDrugDiscovery().to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f'\n Model Summary')
print(f'   Total parameters    : {total:>12,}')
print(f'   Trainable           : {trainable:>12,}')
print(f'   Frozen (ESM-2)      : {frozen:>12,}')

# Forward pass smoke test
model.eval()
with torch.no_grad():
    _b = sample_batch
    _aff, _logits, _fused = model(
        _b['graph_2d'].to(DEVICE),
        _b['graph_3d'].to(DEVICE),
        _b['prot_ids'].to(DEVICE),
        _b['prot_mask'].to(DEVICE),
        smiles_tgt=_b['smiles_enc'][:, :-1].to(DEVICE)
    )
print(f'\n Forward pass OK')
print(f'   affinity shape : {_aff.shape}')
print(f'   logits shape   : {_logits.shape}')
print(f'   fused shape    : {_fused.shape}')

---
##  Cell 21 — Loss Function

In [ ]:
class CombinedLoss(nn.Module):
    """
    Weighted sum of affinity (Huber) and generation (cross-entropy) losses.

    FIX: losses are normalized before weighting so alpha truly controls the
    balance.  Raw gen loss (CE over 27 tokens) is ~15-25x larger than the
    affinity Huber loss, so naive weighting let generation dominate.
    We scale gen_loss by (aff_loss.detach() / gen_loss.detach()) each step
    so both terms contribute equally before alpha is applied.
    """

    def __init__(self, alpha: float = ALPHA, label_smoothing: float = 0.05):
        super().__init__()
        self.alpha           = alpha
        self.huber           = nn.HuberLoss(delta=0.1)
        self.label_smoothing = label_smoothing
        # Running estimates of each loss magnitude for normalisation
        self.register_buffer('_aff_ema', torch.tensor(1.0))
        self.register_buffer('_gen_ema', torch.tensor(1.0))
        self._ema_decay = 0.98

    def forward(
        self,
        aff_pred:   torch.Tensor,
        aff_target: torch.Tensor,
        smi_logits: torch.Tensor,
        smi_target: torch.Tensor,
    ):
        loss_aff = self.huber(aff_pred, aff_target)

        B, T, V = smi_logits.shape
        loss_gen = F.cross_entropy(
            smi_logits.reshape(B * T, V),
            smi_target.reshape(B * T),
            ignore_index=PAD_IDX,
            label_smoothing=self.label_smoothing
        )

        # Update EMA of each loss magnitude (detached — no gradient through this)
        if self.training:
            with torch.no_grad():
                self._aff_ema = self._ema_decay * self._aff_ema + (1 - self._ema_decay) * loss_aff.detach()
                self._gen_ema = self._ema_decay * self._gen_ema + (1 - self._ema_decay) * loss_gen.detach()

        # Scale gen_loss so both are on the same magnitude, then apply alpha
        aff_scale = self._aff_ema.clamp(min=1e-6)
        gen_scale = self._gen_ema.clamp(min=1e-6)
        loss_aff_n = loss_aff / aff_scale
        loss_gen_n = loss_gen / gen_scale

        total = self.alpha * loss_aff_n + (1.0 - self.alpha) * loss_gen_n
        return total, loss_aff, loss_gen


criterion = CombinedLoss(alpha=ALPHA)
print(f' CombinedLoss (normalised) — alpha={ALPHA} (affinity) / {1-ALPHA:.1f} (generation)')


---
##  Cell 22 — Optimiser & Scheduler

In [ ]:
# ── Separate learning rates for PLM (fine-tuned) vs rest ─────────────────────
plm_params   = list(model.prot_encoder.esm.parameters())
plm_ids      = {id(p) for p in plm_params}
other_params = [p for p in model.parameters() if id(p) not in plm_ids and p.requires_grad]

param_groups = [
    {'params': other_params, 'lr': LR},
]
# FIX: PLM is now unfrozen — use 10x lower LR to avoid catastrophic forgetting
if not FREEZE_PLM:
    plm_trainable = [p for p in plm_params if p.requires_grad]
    param_groups.append({'params': plm_trainable, 'lr': LR * 0.1})
    print(f'   PLM fine-tuning ON — PLM LR={LR * 0.1:.1e}, rest LR={LR:.1e}')

optimizer = torch.optim.AdamW(
    param_groups,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8
)

# FIX: T_0=25 (was 10) so the LR doesn't restart mid-training before the model
# has converged. With patience=10 the old T_0=10 caused a restart right when
# early-stopping was counting down, wasting epochs.
scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0=25,         # was 10 — restart every 25 epochs
    T_mult=2,
    eta_min=1e-6
)

print(f' Optimiser: AdamW — LR={LR}, WD={WEIGHT_DECAY}')
print(f'   Parameter groups: {len(param_groups)}')
print(f'   Non-PLM trainable: {sum(p.numel() for p in other_params):,}')
if not FREEZE_PLM:
    print(f'   PLM trainable    : {sum(p.numel() for p in plm_params if p.requires_grad):,}')


---
##  Cell 23 — Train & Evaluate Step Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    totals = {'loss': 0.0, 'aff': 0.0, 'gen': 0.0}
    n_batches = 0

    for batch in tqdm(loader, desc='  train', leave=False):
        if batch is None: continue

        g2d  = batch['graph_2d'].to(DEVICE)
        g3d  = batch['graph_3d'].to(DEVICE)
        ids  = batch['prot_ids'].to(DEVICE)
        mask = batch['prot_mask'].to(DEVICE)
        aff  = batch['affinity'].to(DEVICE)
        enc  = batch['smiles_enc'].to(DEVICE)    # (B, MAX_SMILES_LEN)

        # Teacher forcing: input = enc[:, :-1], target = enc[:, 1:]
        tgt_in  = enc[:, :-1]
        tgt_out = enc[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        aff_pred, smi_logits, _ = model(g2d, g3d, ids, mask, smiles_tgt=tgt_in)

        loss, l_aff, l_gen = criterion(aff_pred, aff, smi_logits, tgt_out)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        totals['loss'] += loss.item()
        totals['aff']  += l_aff.item()
        totals['gen']  += l_gen.item()
        n_batches += 1

    return {k: v / max(n_batches, 1) for k, v in totals.items()}


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    totals = {'loss': 0.0, 'aff': 0.0, 'gen': 0.0}
    all_preds, all_targets = [], []
    n_batches = 0

    for batch in tqdm(loader, desc='  eval ', leave=False):
        if batch is None: continue

        g2d  = batch['graph_2d'].to(DEVICE)
        g3d  = batch['graph_3d'].to(DEVICE)
        ids  = batch['prot_ids'].to(DEVICE)
        mask = batch['prot_mask'].to(DEVICE)
        aff  = batch['affinity'].to(DEVICE)
        enc  = batch['smiles_enc'].to(DEVICE)

        tgt_in  = enc[:, :-1]
        tgt_out = enc[:, 1:]

        aff_pred, smi_logits, _ = model(g2d, g3d, ids, mask, smiles_tgt=tgt_in)

        loss, l_aff, l_gen = criterion(aff_pred, aff, smi_logits, tgt_out)

        totals['loss'] += loss.item()
        totals['aff']  += l_aff.item()
        totals['gen']  += l_gen.item()
        n_batches += 1

        all_preds.extend(aff_pred.cpu().numpy())
        all_targets.extend(aff.cpu().numpy())

    avg = {k: v / max(n_batches, 1) for k, v in totals.items()}
    return avg, np.array(all_preds), np.array(all_targets)


print(' train_epoch and eval_epoch defined.')

---
##  Cell 24 — Metrics

In [ ]:
def compute_metrics(preds: np.ndarray, targets: np.ndarray) -> dict:
    """
    Standard drug-target affinity prediction metrics:
    MSE, RMSE, Pearson r, Spearman ρ, Concordance Index (CI)
    """
    mse     = mean_squared_error(targets, preds)
    rmse    = np.sqrt(mse)
    pcc, _  = pearsonr(preds, targets)
    scc, _  = spearmanr(preds, targets)
    ci      = concordance_index(targets, preds)

    return {
        'MSE':      round(float(mse),  4),
        'RMSE':     round(float(rmse), 4),
        'Pearson':  round(float(pcc),  4),
        'Spearman': round(float(scc),  4),
        'CI':       round(float(ci),   4),
    }


# Test with random values
_t  = np.random.rand(100)
_p  = _t + np.random.randn(100) * 0.1
print('Test metrics:', compute_metrics(_p, _t))
print(' Metrics function OK.')

---
##  Cell 25 — Full Training Loop

In [ ]:
history = {
    'train_loss': [], 'train_aff': [], 'train_gen': [],
    'val_loss':   [], 'val_aff':   [], 'val_gen':   [],
    'val_ci': [], 'val_rmse': [], 'val_pearson': []
}

best_val_ci  = -1.0
best_epoch   = 0
# FIX: patience increased to 15 (was 10) to give the model more room after
# each LR restart.  With T_0=25 the first restart happens at epoch 25, so
# patience=15 means we stop no earlier than epoch 40 if nothing improves.
patience     = 15
no_improve   = 0
CKPT_PATH    = CKPT_DIR / f'{DATASET}_best.pt'

# FIX: affinity-only warmup for the first 3 epochs so the affinity head
# gets a head-start before generation loss enters the picture.
WARMUP_AFFINITY_EPOCHS = 3

print(f' Training on {DATASET.upper()} for {EPOCHS} epochs')
print(f'   Affinity-only warmup: {WARMUP_AFFINITY_EPOCHS} epochs\n')

for epoch in range(1, EPOCHS + 1):

    # ── Warmup phase: affinity-only (alpha=1.0) ─────────────────────────────
    if epoch <= WARMUP_AFFINITY_EPOCHS:
        criterion.alpha = 1.0
    else:
        criterion.alpha = ALPHA

    # ── Train ──────────────────────────────────────────────────────────────
    tr = train_epoch(model, train_loader, optimizer, criterion)
    scheduler.step(epoch)

    # ── Validate ───────────────────────────────────────────────────────────
    va, val_preds, val_tgts = eval_epoch(model, val_loader, criterion)
    metrics = compute_metrics(val_preds, val_tgts)

    # ── Record ─────────────────────────────────────────────────────────────
    history['train_loss'].append(tr['loss'])
    history['train_aff'].append(tr['aff'])
    history['train_gen'].append(tr['gen'])
    history['val_loss'].append(va['loss'])
    history['val_aff'].append(va['aff'])
    history['val_gen'].append(va['gen'])
    history['val_ci'].append(metrics['CI'])
    history['val_rmse'].append(metrics['RMSE'])
    history['val_pearson'].append(metrics['Pearson'])

    # ── Log ────────────────────────────────────────────────────────────────
    lr_now   = optimizer.param_groups[0]['lr']
    warmup_tag = ' [warmup]' if epoch <= WARMUP_AFFINITY_EPOCHS else ''
    print(
        f'Epoch {epoch:3d}/{EPOCHS}{warmup_tag} | '
        f'Tr Loss {tr["loss"]:.4f} (aff {tr["aff"]:.4f} gen {tr["gen"]:.4f}) | '
        f'Val Loss {va["loss"]:.4f} | '
        f'CI {metrics["CI"]:.4f} RMSE {metrics["RMSE"]:.4f} r {metrics["Pearson"]:.4f} | '
        f'LR {lr_now:.2e}'
    )

    # ── Checkpoint ─────────────────────────────────────────────────────────
    if metrics['CI'] > best_val_ci:
        best_val_ci = metrics['CI']
        best_epoch  = epoch
        no_improve  = 0
        torch.save({
            'epoch':        epoch,
            'model_state':  model.state_dict(),
            'optim_state':  optimizer.state_dict(),
            'best_ci':      best_val_ci,
            'history':      history,
            'vocab':        {'char2idx': CHAR2IDX, 'idx2char': IDX2CHAR},
            'dataset':      DATASET
        }, CKPT_PATH)
        print(f'   Saved best checkpoint (CI={best_val_ci:.4f})')
    else:
        no_improve += 1

    # ── Early stopping ─────────────────────────────────────────────────────
    if no_improve >= patience:
        print(f'\n  Early stopping at epoch {epoch} (no CI improvement for {patience} epochs)')
        break

print(f'\n Training complete! Best CI={best_val_ci:.4f} at epoch {best_epoch}')


---
##  Cell 26 — Plot Training Curves

In [ ]:
E = len(history['train_loss'])
epochs_x = range(1, E + 1)

fig = plt.figure(figsize=(18, 10))
fig.suptitle(f'{DATASET.upper()} Training History', fontsize=15, fontweight='bold')
gs  = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

def plot(ax, y_tr, y_va, label, color):
    ax.plot(epochs_x, y_tr, color=color, lw=2, label='Train')
    ax.plot(epochs_x, y_va, color=color, lw=2, linestyle='--', label='Val')
    ax.set_title(label); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(alpha=0.3)

plot(fig.add_subplot(gs[0,0]), history['train_loss'],    history['val_loss'],    'Total Loss',      '#6366f1')
plot(fig.add_subplot(gs[0,1]), history['train_aff'],     history['val_aff'],     'Affinity Loss',   '#10b981')
plot(fig.add_subplot(gs[0,2]), history['train_gen'],     history['val_gen'],     'Generation Loss', '#f59e0b')

ax_ci  = fig.add_subplot(gs[1,0])
ax_ci.plot(epochs_x, history['val_ci'],      '#3b82f6', lw=2)
ax_ci.axhline(best_val_ci, color='red', linestyle='--', label=f'Best={best_val_ci:.4f}')
ax_ci.set_title('Val Concordance Index (CI)'); ax_ci.legend(); ax_ci.grid(alpha=0.3)

ax_rm = fig.add_subplot(gs[1,1])
ax_rm.plot(epochs_x, history['val_rmse'], '#ef4444', lw=2)
ax_rm.set_title('Val RMSE'); ax_rm.grid(alpha=0.3)

ax_r  = fig.add_subplot(gs[1,2])
ax_r.plot(epochs_x, history['val_pearson'], '#8b5cf6', lw=2)
ax_r.set_title("Val Pearson r"); ax_r.grid(alpha=0.3)

plt.savefig(f'{DATASET}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Training curves saved.')

---
##  Cell 27 — Test Set Evaluation

In [ ]:
# Load best checkpoint
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f' Loaded best checkpoint from epoch {ckpt["epoch"]} (CI={ckpt["best_ci"]:.4f})')

# Evaluate on test set
_, test_preds, test_tgts = eval_epoch(model, test_loader, criterion)
test_metrics = compute_metrics(test_preds, test_tgts)

print(f'\n TEST SET RESULTS — {DATASET.upper()}')
print('─' * 40)
for k, v in test_metrics.items():
    print(f'  {k:12s}: {v}')
print('─' * 40)

---
##  Cell 28 — Prediction vs Ground Truth Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'{DATASET.upper()} — Test Set Predictions', fontsize=13, fontweight='bold')

# Scatter: pred vs true
ax = axes[0]
ax.scatter(test_tgts, test_preds, alpha=0.4, s=8, color='#3b82f6')
lo, hi = min(test_tgts.min(), test_preds.min()), max(test_tgts.max(), test_preds.max())
ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='y=x')
ax.set_xlabel('True Affinity (normalised)')
ax.set_ylabel('Predicted Affinity')
ax.set_title(f'Pearson r={test_metrics["Pearson"]:.3f} | CI={test_metrics["CI"]:.3f}')
ax.legend(); ax.grid(alpha=0.3)

# Residual distribution
ax2 = axes[1]
residuals = test_preds - test_tgts
ax2.hist(residuals, bins=50, color='#10b981', edgecolor='white', alpha=0.85)
ax2.axvline(0, color='red', lw=2, linestyle='--')
ax2.set_xlabel('Residual (pred - true)')
ax2.set_ylabel('Count')
ax2.set_title(f'RMSE={test_metrics["RMSE"]:.4f} | Mean={residuals.mean():.4f}')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DATASET}_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Prediction plot saved.')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.stats import pearsonr
from lifelines.utils import concordance_index

# ── Global style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Serif',   # academic serif font
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'axes.labelsize':    11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.25,
    'grid.linestyle':    '--',
    'lines.linewidth':   2,
    'legend.framealpha': 0.9,
    'legend.fontsize':   9,
    'figure.dpi':        150,
    'savefig.dpi':       300,          # 300 dpi = required for most thesis formats
    'savefig.bbox':      'tight',
})

PALETTE = {
    'train':   '#2563EB',   # blue
    'val':     '#DC2626',   # red
    'ci':      '#16A34A',   # green
    'rmse':    '#9333EA',   # purple
    'pearson': '#D97706',   # amber
    'scatter': '#1E3A5F',
    'resid':   '#065F46',
    'ref':     '#991B1B',
}

E      = len(history['train_loss'])
epochs = range(1, E + 1)

# ════════════════════════════════════════════════════════════
# FIGURE 1 — Training curves  (save as thesis_fig_training.png)
# ════════════════════════════════════════════════════════════
fig1 = plt.figure(figsize=(14, 9))
fig1.suptitle(
    f'Training History — {DATASET.upper()} Dataset',
    fontsize=15, fontweight='bold', y=1.01
)
gs1 = gridspec.GridSpec(2, 3, figure=fig1, hspace=0.45, wspace=0.35)

def _curve(ax, y_tr, y_va, title, ylabel, color_tr, color_va, best_epoch=None):
    ax.plot(epochs, y_tr, color=color_tr, lw=2,   label='Train')
    ax.plot(epochs, y_va, color=color_va, lw=2, ls='--', label='Validation')
    if best_epoch:
        ax.axvline(best_epoch, color='gray', lw=1, ls=':', alpha=0.7,
                   label=f'Best epoch ({best_epoch})')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()

_curve(fig1.add_subplot(gs1[0,0]),
       history['train_loss'], history['val_loss'],
       'Total Loss', 'Loss',
       PALETTE['train'], PALETTE['val'], best_epoch)

_curve(fig1.add_subplot(gs1[0,1]),
       history['train_aff'], history['val_aff'],
       'Affinity Loss (Huber)', 'Huber Loss',
       PALETTE['train'], PALETTE['val'])

_curve(fig1.add_subplot(gs1[0,2]),
       history['train_gen'], history['val_gen'],
       'Generation Loss (CE)', 'Cross-Entropy',
       PALETTE['train'], PALETTE['val'])

# Concordance Index
ax_ci = fig1.add_subplot(gs1[1,0])
ax_ci.plot(epochs, history['val_ci'], color=PALETTE['ci'], lw=2, label='Val CI')
ax_ci.axhline(best_val_ci, color=PALETTE['ref'], lw=1.2, ls='--',
              label=f'Best CI = {best_val_ci:.4f}')
ax_ci.set_title('Concordance Index (CI ↑)')
ax_ci.set_xlabel('Epoch'); ax_ci.set_ylabel('CI')
ax_ci.legend()

# RMSE
ax_rm = fig1.add_subplot(gs1[1,1])
ax_rm.plot(epochs, history['val_rmse'], color=PALETTE['rmse'], lw=2)
ax_rm.set_title('Validation RMSE (↓)')
ax_rm.set_xlabel('Epoch'); ax_rm.set_ylabel('RMSE')

# Pearson r
ax_r = fig1.add_subplot(gs1[1,2])
ax_r.plot(epochs, history['val_pearson'], color=PALETTE['pearson'], lw=2)
ax_r.set_title("Pearson Correlation (r ↑)")
ax_r.set_xlabel('Epoch'); ax_r.set_ylabel("Pearson r")

fig1.savefig(f'thesis_fig_training_{DATASET}.png')
plt.show()
print(f' Saved: thesis_fig_training_{DATASET}.png')


# ════════════════════════════════════════════════════════════
# FIGURE 2 — Test-set predictions  (save as thesis_fig_predictions.png)
# ════════════════════════════════════════════════════════════
fig2, axes = plt.subplots(1, 2, figsize=(12, 5))
fig2.suptitle(
    f'Test-Set Evaluation — {DATASET.upper()}  '
    f'(n = {len(test_preds):,} pairs)',
    fontsize=14, fontweight='bold'
)

# — Scatter: predicted vs true —
ax = axes[0]
ax.scatter(test_tgts, test_preds,
           alpha=0.35, s=10,
           color=PALETTE['scatter'], rasterized=True)
lo = min(test_tgts.min(), test_preds.min()) - 0.02
hi = max(test_tgts.max(), test_preds.max()) + 0.02
ax.plot([lo, hi], [lo, hi], color=PALETTE['ref'],
        lw=1.5, ls='--', label='Ideal (y = x)')
ax.set_xlabel('True Affinity (normalised)')
ax.set_ylabel('Predicted Affinity')
ax.set_title(
    f'Predicted vs True Affinity\n'
    f'Pearson r = {test_metrics["Pearson"]:.4f}   '
    f'CI = {test_metrics["CI"]:.4f}'
)
ax.legend()
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)

# — Residual distribution —
ax2 = axes[1]
residuals = test_preds - test_tgts
ax2.hist(residuals, bins=60,
         color=PALETTE['resid'], edgecolor='white',
         alpha=0.85, linewidth=0.4)
ax2.axvline(0,                color=PALETTE['ref'], lw=2,   ls='--', label='Zero error')
ax2.axvline(residuals.mean(), color=PALETTE['pearson'], lw=1.5, ls=':',
            label=f'Mean = {residuals.mean():.4f}')
ax2.set_xlabel('Residual (predicted − true)')
ax2.set_ylabel('Count')
ax2.set_title(
    f'Residual Distribution\n'
    f'RMSE = {test_metrics["RMSE"]:.4f}   '
    f'MSE = {test_metrics["MSE"]:.4f}'
)
ax2.legend()

fig2.tight_layout()
fig2.savefig(f'thesis_fig_predictions_{DATASET}.png')
plt.show()
print(f' Saved: thesis_fig_predictions_{DATASET}.png')


# ════════════════════════════════════════════════════════════
# FIGURE 3 — Metrics summary table  (save as thesis_fig_metrics.png)
# ════════════════════════════════════════════════════════════
benchmark = [
    ['DeepDTA',             0.878, 0.261, 0.863, 0.194],
    ['GraphDTA (GCN)',      0.880, 0.254, 0.867, 0.183],
    ['GraphDTA (GAT)',      0.882, 0.245, 0.868, 0.179],
    ['MGraphDTA',          0.897, 0.207, 0.890, 0.139],
    ['This model (ours)',
     test_metrics['CI'], test_metrics['MSE'], '—', '—'],
]
col_labels = ['Model', 'DAVIS CI ↑', 'DAVIS MSE ↓', 'KIBA CI ↑', 'KIBA MSE ↓']

fig3, ax3 = plt.subplots(figsize=(11, 2.8))
ax3.axis('off')
tbl = ax3.table(
    cellText=benchmark,
    colLabels=col_labels,
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 2.0)

# Style header row
for col in range(len(col_labels)):
    tbl[(0, col)].set_facecolor('#1E3A5F')
    tbl[(0, col)].set_text_props(color='white', fontweight='bold')

# Highlight our row in light blue
our_row = len(benchmark)
for col in range(len(col_labels)):
    tbl[(our_row, col)].set_facecolor('#DBEAFE')
    tbl[(our_row, col)].set_text_props(fontweight='bold', color='#1E3A5F')

fig3.suptitle(
    f'Benchmark Comparison — {DATASET.upper()}',
    fontsize=13, fontweight='bold', y=1.05
)
fig3.savefig(f'thesis_fig_benchmark_{DATASET}.png')
plt.show()
print(f' Saved: thesis_fig_benchmark_{DATASET}.png')

# ── Print final metrics neatly ────────────────────────────────────────────────
print(f'\n{"="*45}')
print(f'  FINAL TEST METRICS — {DATASET.upper()}')
print(f'{"="*45}')
for k, v in test_metrics.items():
    print(f'  {k:<12}: {v}')
print(f'{"="*45}')

---
##  Cell 29 — Inference: Predict Affinity + Generate Novel Drug

In [ ]:
def run_inference(
    protein_seq: str,
    query_smiles: str,
    temperature:  float = 0.8,
    n_generate:   int   = 5
):
    """
    Given a protein sequence and a reference SMILES:
      1. Predicts normalised binding affinity.
      2. Generates n_generate novel candidate SMILES.
    Returns a dict with results.
    """
    model.eval()

    # ── Build graphs ────────────────────────────────────────────────────────
    g2d = smiles_to_graph_2d(query_smiles)
    g3d = smiles_to_pharmacophore_graph(query_smiles)
    if g2d is None or g3d is None:
        return {'error': 'Invalid SMILES or 3D conformer failed'}

    # Replicate batch for multiple generations
    g2d_batch = Batch.from_data_list([g2d] * n_generate).to(DEVICE)
    g3d_batch = Batch.from_data_list([g3d] * n_generate).to(DEVICE)

    # ── Tokenise protein ────────────────────────────────────────────────────
    enc = tokeniser(
        protein_seq[:MAX_PROT_LEN],
        max_length=MAX_PROT_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    prot_ids  = enc['input_ids'].repeat(n_generate, 1).to(DEVICE)
    prot_mask = enc['attention_mask'].repeat(n_generate, 1).to(DEVICE)

    with torch.no_grad():
        aff_preds, gen_tokens = model.predict_and_generate(
            g2d_batch, g3d_batch, prot_ids, prot_mask, temperature=temperature
        )

    affinity = aff_preds.mean().item()

    # ── Decode and validate ─────────────────────────────────────────────────
    generated_smiles = []
    for toks in gen_tokens:
        smi = decode_smiles(toks.cpu())
        mol = Chem.MolFromSmiles(smi)
        generated_smiles.append({
            'smiles': smi,
            'valid':  mol is not None
        })

    return {
        'query_smiles':     query_smiles,
        'predicted_affinity': round(affinity, 4),
        'generated':        generated_smiles
    }


# ── Run inference ───────────────────────────────────────────────────────────
sample_row = test_df.iloc[0]
result = run_inference(
    protein_seq   = sample_row['protein_seq'],
    query_smiles  = sample_row['smiles'],
    temperature   = 0.8,
    n_generate    = 5
)

print('=== INFERENCE RESULTS ===')
print(f"Reference SMILES   : {result['query_smiles']}")
print(f"True affinity (norm): {sample_row['affinity_norm']:.4f}")
print(f"Predicted affinity  : {result['predicted_affinity']}")
print(f"\nGenerated candidates:")
for i, g in enumerate(result['generated'], 1):
    flag = '' if g['valid'] else '❌'
    print(f"  {i}. {flag} {g['smiles']}")

---
##  Cell 30 — Visualise Generated Molecules

In [ ]:
from rdkit.Chem import Draw
from PIL import Image
import io

valid_mols = [
    (g['smiles'], Chem.MolFromSmiles(g['smiles']))
    for g in result['generated'] if g['valid']
]

if valid_mols:
    ref_mol = Chem.MolFromSmiles(result['query_smiles'])
    all_mols   = [ref_mol]    + [m for _, m in valid_mols]
    all_labels = ['Reference'] + [f'Generated {i+1}' for i in range(len(valid_mols))]

    img = Draw.MolsToGridImage(
        all_mols,
        molsPerRow=3,
        subImgSize=(350, 280),
        legends=all_labels,
        returnPNG=False,   # FIX: ensure we get a PIL Image, not bytes
    )
    # FIX: MolsToGridImage may return bytes in some RDKit versions;
    # handle both cases so .save() always works.
    if isinstance(img, bytes):
        img = Image.open(io.BytesIO(img))
    img.save('generated_molecules.png')
    display(img)
    print(f' {len(valid_mols)} valid molecules generated and displayed.')
else:
    print('⚠️ No valid molecules generated. Try adjusting temperature or training longer.')


---
##  Cell 31 — Save Final Checkpoint & Export

In [ ]:
final_ckpt_path = CKPT_DIR / f'{DATASET}_final.pt'

torch.save({
    'model_state':    model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'history':        history,
    'test_metrics':   test_metrics,
    'best_val_ci':    best_val_ci,
    'best_epoch':     best_epoch,
    'dataset':        DATASET,
    'config': {
        'PLM_NAME':    PLM_NAME,
        'MAX_PROT_LEN': MAX_PROT_LEN,
        'MODAL_DIM':   MODAL_DIM,
        'FUSION_DIM':  FUSION_DIM,
        'MAX_SMILES_LEN': MAX_SMILES_LEN,
    },
    'vocab': {
        'char2idx': CHAR2IDX,
        'idx2char': IDX2CHAR,
        'vocab_size': VOCAB_SIZE
    }
}, final_ckpt_path)

print(f' Final checkpoint saved: {final_ckpt_path}')

# ── Summary ──────────────────────────────────────────────────────────────────
print(f'\n{'='*50}')
print(f'  FINAL SUMMARY — {DATASET.upper()}')
print(f'{'='*50}')
print(f'  Best Val CI   : {best_val_ci:.4f}  (epoch {best_epoch})')
print(f'  Test CI       : {test_metrics["CI"]}')
print(f'  Test RMSE     : {test_metrics["RMSE"]}')
print(f'  Test Pearson r: {test_metrics["Pearson"]}')
print(f'  Test Spearman : {test_metrics["Spearman"]}')
print(f'{'='*50}')

---
##  Cell 32 — Switch to KIBA Dataset (Optional)
> Re-run from Cell 3 with `DATASET = 'kiba'`, or use this convenience cell.

In [ ]:
# ── Switch dataset ────────────────────────────────────────────────────────────
DATASET = 'kiba'

df_kiba   = load_dataset('kiba')
df_kiba   = normalise_affinity(df_kiba, 'kiba')
df_kiba   = df_kiba[df_kiba['smiles'].apply(is_valid_smiles)].copy()

train_kiba = df_kiba[df_kiba['split'] == 'train'].reset_index(drop=True)
test_kiba  = df_kiba[df_kiba['split'] == 'test' ].reset_index(drop=True)

# FIX: random val split (was sequential head which caused distribution mismatch)
val_size   = int(0.1 * len(train_kiba))
val_idx    = train_kiba.sample(val_size, random_state=SEED).index
val_kiba   = train_kiba.loc[val_idx].copy()
train_kiba = train_kiba.drop(val_idx).copy()

train_loader_kiba = DataLoader(DrugTargetDataset(train_kiba, tokeniser),
    batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader_kiba   = DataLoader(DrugTargetDataset(val_kiba,   tokeniser),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader_kiba  = DataLoader(DrugTargetDataset(test_kiba,  tokeniser),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f' KIBA splits — Train: {len(train_kiba):,} | Val: {len(val_kiba):,} | Test: {len(test_kiba):,}')
print(f'   Affinity range: [{df_kiba.affinity_norm.min():.3f}, {df_kiba.affinity_norm.max():.3f}]')
print('\n  Re-initialise the model and run Cell 25 with kiba loaders to train on KIBA.')


---
##  Cell 33 — Benchmark Comparison Table

In [ ]:
# Reference numbers from published papers (DeepDTA, GraphDTA, MGraphDTA, etc.)
benchmark = pd.DataFrame([
    {'Model': 'DeepDTA',             'DAVIS CI': 0.878, 'DAVIS MSE': 0.261, 'KIBA CI': 0.863, 'KIBA MSE': 0.194},
    {'Model': 'GraphDTA (GCN)',       'DAVIS CI': 0.880, 'DAVIS MSE': 0.254, 'KIBA CI': 0.867, 'KIBA MSE': 0.183},
    {'Model': 'GraphDTA (GAT)',       'DAVIS CI': 0.882, 'DAVIS MSE': 0.245, 'KIBA CI': 0.868, 'KIBA MSE': 0.179},
    {'Model': 'MGraphDTA',            'DAVIS CI': 0.897, 'DAVIS MSE': 0.207, 'KIBA CI': 0.890, 'KIBA MSE': 0.139},
    {'Model': '⭐ This Model (Ours)', 'DAVIS CI': test_metrics['CI'], 'DAVIS MSE': test_metrics['MSE'], 'KIBA CI': '-', 'KIBA MSE': '-'},
])

print('\n📊 Benchmark Comparison:')
print(benchmark.to_string(index=False))

# Visual table
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')
table = ax.table(
    cellText=benchmark.values,
    colLabels=benchmark.columns,
    cellLoc='center', loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)
# Highlight our row
for col in range(len(benchmark.columns)):
    table[(len(benchmark), col)].set_facecolor('#dbeafe')
    table[(len(benchmark), col)].set_text_props(fontweight='bold')
plt.title('Benchmark Comparison — DAVIS & KIBA', fontweight='bold', pad=12)
plt.savefig('benchmark_table.png', dpi=150, bbox_inches='tight')
plt.show()